# 1_silver_transferencias_autos
**Capa Silver de transferencias de autos 2026 (DNRPA)** · adaptado a lo que hallamos en el EDA.

**Flujo:** Bronze (todo texto) → vista temporal de prueba (`transf_silver_prep`) → tabla Silver con tipos reales → `MERGE` incremental.

**Reglas que aplica (decisiones del EDA):**
- No se borra ninguna fila. Lo raro se **marca** en `dq_problemas` y los valores imposibles quedan en NULL en la columna derivada.
- `COUNT(*)` = transferencias registradas (una fila = una titularidad adquirida). No se usa `peso`.
- Flotas de personas jurídicas (filas idénticas) se **conservan**; se numeran con `repeticion` y se cuentan en `filas_identicas`.
- Personas jurídicas: `titular_anio_nacimiento` = NULL. `No aplica` en género y país se mantiene.

## Parte 0: Verificar Bronze

In [0]:
%sql
USE CATALOG bootcamp;
SELECT COUNT(*) AS filas_bronze, COUNT(DISTINCT _archivo_origen) AS archivos
FROM bootcamp.bronze.transferencias_autos_bronze;
-- Esperado: 1.378.157 filas y 9 archivos

## Parte 1: Crear la tabla Silver (estructura primero, como en propiedades)

Definimos la tabla **con tipos explícitos** antes de cargar. Si algún día el SELECT devuelve un tipo distinto, el `MERGE` falla (queremos que falle, no que guarde basura).

> Ojo: este bloque hace `DROP TABLE IF EXISTS`. Si ya cargaste la tabla y querés conservarla, no lo corras de nuevo.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bootcamp.silver COMMENT 'Capa Silver: datos limpios, tipados y validados';

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.transferencias_autos;

CREATE TABLE bootcamp.silver.transferencias_autos (
  -- Tiempo
  periodo_archivo STRING   COMMENT 'AAAAMM sacado del nombre del archivo',
  mes DATE                 COMMENT 'Primer día del mes del trámite',
  tramite_fecha DATE       COMMENT 'Fecha del trámite (DATE)',
  -- Trámite
  tramite_tipo STRING      COMMENT 'Tipo de trámite tal como viene (limpio de espacios)',
  rol_comerciante STRING   COMMENT 'COMERCIANTE_COMPRA / COMERCIANTE_VENDE / SIN_COMERCIANTE',
  -- Registro
  registro_seccional_id STRING          COMMENT 'Código del registro seccional (texto, no es un número para sumar)',
  registro_seccional_descripcion STRING COMMENT 'Nombre del registro seccional',
  registro_seccional_provincia STRING   COMMENT 'Provincia del registro (NULL en 1,6% de las filas)',
  -- Automotor
  automotor_origen STRING             COMMENT 'Nacional / Importado / Protocolo 21',
  automotor_anio_modelo INT           COMMENT 'Año modelo; NULL si vacío o fuera de rango',
  fecha_inscripcion_inicial DATE      COMMENT 'Inscripción inicial; NULL si vacía',
  antiguedad_anios DECIMAL(5,1)       COMMENT 'Antigüedad del auto AL MOMENTO del trámite (no la actual)',
  automotor_tipo_id STRING            COMMENT 'Código de tipo de vehículo',
  automotor_tipo_descripcion STRING   COMMENT 'Tipo de vehículo normalizado (PTAS->PUERTAS, PICK UP->PICK-UP)',
  categoria_vehiculo STRING           COMMENT 'AUTO / PICK-UP / UTILITARIO / PESADO / REMOLQUE / OTRO',
  automotor_marca STRING              COMMENT 'Marca normalizada (typos corregidos). USAR ESTA para agrupar',
  automotor_marca_original STRING     COMMENT 'Marca como vino (solo mayúsculas y espacios limpios)',
  automotor_marca_codigo_original STRING COMMENT 'Código de marca del origen. NO confiable como ID',
  automotor_modelo_codigo_original STRING COMMENT 'Código de modelo del origen. NO confiable como ID',
  automotor_modelo_descripcion STRING COMMENT 'Modelo completo',
  modelo_base STRING                  COMMENT 'Primera palabra del modelo (aproximación)',
  automotor_uso_id STRING             COMMENT 'Código de uso',
  automotor_uso_descripcion STRING    COMMENT 'Privado / Público / Oficial / No declarado',
  -- Titular
  titular_tipo_persona STRING         COMMENT 'Física / Jurídica',
  titular_genero STRING               COMMENT 'Género; "No aplica" en jurídicas (se mantiene)',
  titular_anio_nacimiento INT         COMMENT 'Solo personas físicas; NULL en jurídicas',
  edad_titular INT                    COMMENT 'Edad al trámite (solo físicas, 18 a 110)',
  titular_pais_nacimiento STRING      COMMENT 'País; "No aplica" en jurídicas (se mantiene)',
  titular_pais_nacimiento_id STRING   COMMENT 'ID de país (vacío en 6%)',
  titular_domicilio_localidad STRING,
  titular_domicilio_provincia STRING,
  titular_domicilio_provincia_id STRING,
  titular_porcentaje_titularidad DOUBLE COMMENT 'Porcentaje original que adquirió el titular',
  es_transferencia_parcial BOOLEAN    COMMENT 'TRUE si el porcentaje < 100 (copropiedad)',
  -- Control de calidad y llave técnica
  hash_fila STRING        COMMENT 'SHA-256 de las 25 columnas de Bronze (identifica filas idénticas)',
  repeticion INT          COMMENT '1,2,3... entre filas idénticas (flotas). Llave técnica = hash_fila + repeticion',
  filas_identicas INT     COMMENT 'Cuántas filas idénticas hay en total (1 = única)',
  dq_problemas STRING     COMMENT 'Lista de problemas separados por coma; vacío = sin problemas',
  fila_valida BOOLEAN     COMMENT 'TRUE si dq_problemas está vacío',
  -- Metadata
  _archivo_origen STRING,
  _ingesta_ts TIMESTAMP,
  _processing_timestamp TIMESTAMP COMMENT 'Cuándo se cargó en Silver'
)
USING DELTA
COMMENT 'Transferencias de automotores DNRPA 2026 - Capa Silver';

## Parte 2: Vista temporal de prueba (la "prueba en seco")

Igual que en propiedades: primero armamos la transformación como **vista**, la revisamos con `SELECT`, y recién después la insertamos. Así un error de conversión no ensucia la tabla.

### Qué herramienta usamos para cada cosa

| Necesidad | Herramienta | Ejemplo en esta tabla |
|---|---|---|
| Convertir texto a número/fecha **sin juzgar** (si no se puede, NULL) | `TRY_CAST` / `try_to_date` | `automotor_anio_modelo`, `tramite_fecha`, porcentaje |
| Aplicar una **regla de negocio** (rango, categoría, traducir códigos) | `CASE WHEN` | `categoria_vehiculo`, `rol_comerciante`, marca, edad 18-110 |
| Texto vacío o con espacios → NULL | `NULLIF(TRIM(col), '')` | todas las columnas de texto |
| Unificar variantes de escritura | `UPPER` + `regexp_replace` | `PTAS`→`PUERTAS`, `PICK UP`→`PICK-UP` |
| Marcar problemas sin borrar | `filter(array(CASE...), x -> x IS NOT NULL)` | `dq_problemas` |

**Por qué `try_to_date(x, 'yyyy-MM-dd')` y no `TRY_CAST(x AS DATE)`:** exige el formato exacto; si mañana llega `18/09/2026` queda NULL y lo marcamos, en vez de interpretarlo mal.

**Por qué TRY_CAST y no CAST:** `CAST` rompe toda la carga ante un solo valor feo. `TRY_CAST` lo deja en NULL y el flag lo informa. En el EDA ya vimos que hoy hay **0 fallas**, pero la carga de mañana puede traer sorpresas.

**TRY_CAST + CASE WHEN juntos:** primero convierto (`TRY_CAST`), después juzgo (`CASE WHEN ... BETWEEN`). Son dos pasos distintos y por eso van en capas (CTE) separadas.

### Capas de la vista (CTE)
`b` hash de la fila → `p` limpieza de texto → `q` tipado → `r` derivadas y reglas → `s` flags de calidad → SELECT final con tipos explícitos.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW transf_silver_prep AS
WITH b AS (
  -- CAPA b: huella de la fila sobre el TEXTO ORIGINAL de las 25 columnas (antes de limpiar).
  -- to_json(struct(...)) arma un texto con "nombre":"valor" por cada campo. Los NULL no necesitan COALESCE:
  -- como cada valor viaja con el nombre de su columna, (genero=NULL, pais_id=32) y (genero=32, pais_id=NULL)
  -- dan textos distintos. Los NULL siguen siendo NULL de verdad en todas las columnas de la vista.
  SELECT *,
    sha2(to_json(struct(
      tramite_tipo,
      tramite_fecha,
      fecha_inscripcion_inicial,
      registro_seccional_codigo,
      registro_seccional_descripcion,
      registro_seccional_provincia,
      automotor_origen,
      automotor_anio_modelo,
      automotor_tipo_codigo,
      automotor_tipo_descripcion,
      automotor_marca_codigo,
      automotor_marca_descripcion,
      automotor_modelo_codigo,
      automotor_modelo_descripcion,
      automotor_uso_codigo,
      automotor_uso_descripcion,
      titular_tipo_persona,
      titular_domicilio_localidad,
      titular_domicilio_provincia,
      titular_genero,
      titular_anio_nacimiento,
      titular_pais_nacimiento,
      titular_porcentaje_titularidad,
      titular_domicilio_provincia_id,
      titular_pais_nacimiento_id
    )), 256) AS hash_fila
  FROM bootcamp.bronze.transferencias_autos_bronze
),
p AS (
  -- CAPA p: LIMPIEZA DE TEXTO (todavía todo STRING)
  -- NULLIF(TRIM(x), '') convierte '' y '   ' en NULL (en el EDA vimos 8 fechas de inscripción con solo espacios).
  SELECT
    hash_fila, _archivo_origen, _ingesta_ts,
    regexp_extract(_archivo_origen, '([0-9]{6})', 1)                     AS periodo_archivo,
    NULLIF(TRIM(tramite_tipo), '')                                        AS tramite_tipo,
    NULLIF(TRIM(tramite_fecha), '')                                       AS tramite_fecha_txt,
    NULLIF(TRIM(fecha_inscripcion_inicial), '')                           AS inscripcion_txt,
    NULLIF(TRIM(registro_seccional_codigo), '')                           AS registro_seccional_id,
    NULLIF(TRIM(registro_seccional_descripcion), '')                      AS registro_seccional_descripcion,
    NULLIF(TRIM(registro_seccional_provincia), '')                        AS registro_seccional_provincia,
    NULLIF(TRIM(automotor_origen), '')                                    AS automotor_origen,
    NULLIF(TRIM(automotor_anio_modelo), '')                               AS anio_modelo_txt,
    NULLIF(TRIM(automotor_tipo_codigo), '')                               AS automotor_tipo_id,
    -- Normalizar variantes de escritura: 'SEDAN 5 PTAS' -> 'SEDAN 5 PUERTAS', 'PICK UP' -> 'PICK-UP'
    -- (regexp_replace y no un CASE porque son patrones que se repiten en muchos valores)
    regexp_replace(regexp_replace(UPPER(TRIM(automotor_tipo_descripcion)), ' PTAS$', ' PUERTAS'),
                   'PICK UP', 'PICK-UP')                                  AS automotor_tipo_descripcion,
    NULLIF(TRIM(automotor_marca_codigo), '')                              AS automotor_marca_codigo_original,
    -- Marca: mayúsculas y espacios dobles unificados. La corrección de typos se hace en la capa r.
    regexp_replace(UPPER(TRIM(automotor_marca_descripcion)), '[ ]+', ' ') AS marca_txt,
    NULLIF(TRIM(automotor_modelo_codigo), '')                             AS automotor_modelo_codigo_original,
    regexp_replace(UPPER(TRIM(automotor_modelo_descripcion)), '[ ]+', ' ') AS automotor_modelo_descripcion,
    NULLIF(TRIM(automotor_uso_codigo), '')                                AS automotor_uso_id,
    NULLIF(TRIM(automotor_uso_descripcion), '')                           AS automotor_uso_descripcion,
    NULLIF(TRIM(titular_tipo_persona), '')                                AS titular_tipo_persona,
    NULLIF(TRIM(titular_domicilio_localidad), '')                         AS titular_domicilio_localidad,
    NULLIF(TRIM(titular_domicilio_provincia), '')                         AS titular_domicilio_provincia,
    NULLIF(TRIM(titular_domicilio_provincia_id), '')                      AS titular_domicilio_provincia_id,
    -- 'No aplica' se MANTIENE en género y país (decisión: sirve para separar las jurídicas)
    NULLIF(TRIM(titular_genero), '')                                      AS titular_genero,
    NULLIF(TRIM(titular_anio_nacimiento), '')                             AS anio_nac_txt,
    NULLIF(TRIM(titular_pais_nacimiento), '')                             AS titular_pais_nacimiento,
    NULLIF(TRIM(titular_pais_nacimiento_id), '')                          AS titular_pais_nacimiento_id,
    NULLIF(TRIM(titular_porcentaje_titularidad), '')                      AS pct_txt
  FROM b
),
q AS (
  -- CAPA q: TIPADO. Solo convertir, sin juzgar. Si no convierte -> NULL.
  SELECT *,
    try_to_date(tramite_fecha_txt, 'yyyy-MM-dd')  AS tramite_fecha,
    try_to_date(inscripcion_txt,   'yyyy-MM-dd')  AS fecha_inscripcion_inicial,
    TRY_CAST(anio_modelo_txt AS INT)              AS anio_modelo_raw,
    TRY_CAST(anio_nac_txt AS INT)                 AS anio_nac_raw,
    TRY_CAST(pct_txt AS DOUBLE)                   AS titular_porcentaje_titularidad
  FROM p
),
r AS (
  -- CAPA r: REGLAS DE NEGOCIO (CASE WHEN) y columnas derivadas
  SELECT *,
    TRUNC(tramite_fecha, 'MM') AS mes,
    split(automotor_modelo_descripcion, ' ')[0] AS modelo_base,   -- aproximación: 'GRAND VITARA' queda como 'GRAND'

    -- Año modelo: se acepta 1930 .. (año del trámite + 1). Fuera de eso -> NULL (y flag en la capa s).
    -- Vacío también queda NULL: la fila se conserva porque aporta información en otras columnas.
    CASE WHEN anio_modelo_raw BETWEEN 1930 AND YEAR(tramite_fecha) + 1 THEN anio_modelo_raw END AS automotor_anio_modelo,

    -- Año de nacimiento: SOLO personas físicas. En jurídicas el dato no significa nada (hay 1900 a 9999).
    CASE WHEN titular_tipo_persona = 'Física' THEN anio_nac_raw END AS titular_anio_nacimiento,

    -- Edad al trámite (aprox., año contra año): solo físicas y solo entre 18 y 110
    CASE WHEN titular_tipo_persona = 'Física'
          AND YEAR(tramite_fecha) - anio_nac_raw BETWEEN 18 AND 110
         THEN YEAR(tramite_fecha) - anio_nac_raw END AS edad_titular,

    -- Antigüedad del auto al momento del trámite. Se calcula solo si la inscripción es plausible:
    -- >= 1930 (descarta el 1900-01-01 de relleno, que daba 126 años) y no posterior al trámite.
    CASE WHEN fecha_inscripcion_inicial >= DATE'1930-01-01'
          AND fecha_inscripcion_inicial <= tramite_fecha
         THEN ROUND(DATEDIFF(tramite_fecha, fecha_inscripcion_inicial) / 365.25, 1) END AS antiguedad_anios,

    -- Rol del comerciante habitualista (concesionaria). Los LIKE llevan espacio antes de 'A'/'DE'
    -- para no confundir con otras palabras; hay variantes con doble espacio ('NACIONAL  DE COM. HAB.').
    CASE
      WHEN regexp_replace(tramite_tipo, '[ ]+', ' ') LIKE '% A COM. HAB%'  THEN 'COMERCIANTE_COMPRA'
      WHEN regexp_replace(tramite_tipo, '[ ]+', ' ') LIKE '% DE COM. HAB%' THEN 'COMERCIANTE_VENDE'
      ELSE 'SIN_COMERCIANTE'
    END AS rol_comerciante,

    -- Copropiedad: guardamos el porcentaje original y una bandera (NO usamos 'peso' para contar)
    titular_porcentaje_titularidad < 100 AS es_transferencia_parcial,

    -- Marca normalizada: la descripción manda (el código de marca NO es confiable: FORD tiene 92 códigos).
    CASE
      WHEN marca_txt IN ('SIN ESPECIFICACION', 'SIN MARCA')                       THEN NULL
      WHEN marca_txt LIKE 'FORD%' AND marca_txt <> 'FORD FALCON'                  THEN 'FORD'
      WHEN marca_txt LIKE 'VOLKSWAGEN%' OR marca_txt = 'VOLSKWAGEN'               THEN 'VOLKSWAGEN'
      WHEN marca_txt LIKE 'MERCEDES%' OR marca_txt = 'M BENZ'                     THEN 'MERCEDES BENZ'
      WHEN marca_txt LIKE 'CHEVROLET%' OR marca_txt LIKE '%CHEVROLET'
           OR marca_txt IN ('CHEVROELT', 'CEHVROLET')                             THEN 'CHEVROLET'
      WHEN marca_txt LIKE 'RENAULT%' OR marca_txt IN ('RENUALT', 'RENAUL', 'RANAULT') THEN 'RENAULT'
      WHEN marca_txt IN ('PEUGEOT', 'PUEGEOT')                                    THEN 'PEUGEOT'
      WHEN marca_txt LIKE 'TOYOTA%' OR marca_txt = 'TOYOYA'                       THEN 'TOYOTA'
      WHEN marca_txt IN ('BMW', 'B M W', 'B.M.W.')                                THEN 'BMW'
      ELSE marca_txt
    END AS automotor_marca,

    -- Categoría de vehículo (el EDA mostró que el dataset incluye pick-ups, utilitarios y pesados)
    CASE
      WHEN automotor_tipo_descripcion = 'SIN ESPECIFICACION' THEN NULL
      WHEN automotor_tipo_descripcion RLIKE '^(SEDAN|RURAL|TODO TERRENO|FAMILIAR|COUPE|BERLINA|AUTOMOVIL)' THEN 'AUTO'
      WHEN automotor_tipo_descripcion LIKE 'PICK-UP%'                       THEN 'PICK-UP'
      WHEN automotor_tipo_descripcion RLIKE '^(FURGON|UTILITARIO)'          THEN 'UTILITARIO'
      WHEN automotor_tipo_descripcion RLIKE '^(CAMION|TRACTOR|CHASIS|OMNIBUS|TRANSP)'
           OR automotor_tipo_descripcion IN ('ACOPLADO', 'SEMIRREMOLQUE')    THEN 'PESADO / REMOLQUE'
      ELSE 'OTRO'   -- revisar con la query V6
    END AS categoria_vehiculo
  FROM q
),
s AS (
  -- CAPA s: FLAGS DE CALIDAD. Una lista de problemas por fila. No borra nada.
  -- Truco: array(CASE...) arma la lista; filter(...) saca los NULL; array_join la convierte en texto.
  SELECT *,
    array_join(filter(array(
      CASE WHEN tramite_fecha_txt IS NULL THEN 'tramite_fecha_vacia'
           WHEN tramite_fecha IS NULL     THEN 'tramite_fecha_formato_invalido' END,
      CASE WHEN tramite_fecha IS NOT NULL
            AND date_format(tramite_fecha, 'yyyyMM') <> periodo_archivo THEN 'tramite_fecha_fuera_del_periodo_del_archivo' END,
      CASE WHEN inscripcion_txt IS NULL THEN 'inscripcion_vacia'
           WHEN fecha_inscripcion_inicial IS NULL THEN 'inscripcion_formato_invalido'
           WHEN fecha_inscripcion_inicial < DATE'1930-01-01' THEN 'inscripcion_anterior_a_1930'
           WHEN fecha_inscripcion_inicial > tramite_fecha    THEN 'inscripcion_posterior_al_tramite' END,
      CASE WHEN anio_modelo_txt IS NULL THEN 'anio_modelo_vacio'
           WHEN anio_modelo_raw IS NULL THEN 'anio_modelo_formato_invalido'
           WHEN automotor_anio_modelo IS NULL THEN 'anio_modelo_fuera_de_rango' END,
      CASE WHEN titular_tipo_persona = 'Física' AND anio_nac_txt IS NULL THEN 'anio_nacimiento_vacio'
           WHEN titular_tipo_persona = 'Física' AND anio_nac_raw IS NOT NULL AND edad_titular IS NULL
                THEN 'edad_titular_fuera_de_rango' END,
      CASE WHEN pct_txt IS NULL THEN 'porcentaje_vacio'
           WHEN titular_porcentaje_titularidad IS NULL THEN 'porcentaje_formato_invalido'
           WHEN titular_porcentaje_titularidad <= 0 OR titular_porcentaje_titularidad > 100 THEN 'porcentaje_fuera_de_rango' END,
      CASE WHEN marca_txt IS NULL OR marca_txt = '' THEN 'marca_vacia' END,
      CASE WHEN automotor_modelo_descripcion IS NULL OR automotor_modelo_descripcion = '' THEN 'modelo_vacio' END,
      CASE WHEN automotor_origen IS NULL OR automotor_origen NOT IN ('Nacional', 'Importado', 'Protocolo 21') THEN 'origen_inesperado' END,
      CASE WHEN titular_tipo_persona IS NULL OR titular_tipo_persona NOT IN ('Física', 'Jurídica') THEN 'tipo_persona_inesperado' END,
      CASE WHEN automotor_uso_descripcion IS NULL THEN 'uso_vacio' END,
      CASE WHEN registro_seccional_provincia IS NULL THEN 'provincia_registro_vacia' END
    ), x -> x IS NOT NULL), ',') AS dq_problemas
  FROM r
)
SELECT
  -- SELECT final: CAST explícito a los tipos de la tabla Silver (si algo no calza, falla acá y no en silencio)
  CAST(periodo_archivo AS STRING)                    AS periodo_archivo,
  CAST(mes AS DATE)                                  AS mes,
  CAST(tramite_fecha AS DATE)                        AS tramite_fecha,
  tramite_tipo,
  rol_comerciante,
  registro_seccional_id, registro_seccional_descripcion, registro_seccional_provincia,
  automotor_origen,
  CAST(automotor_anio_modelo AS INT)                 AS automotor_anio_modelo,
  CAST(fecha_inscripcion_inicial AS DATE)            AS fecha_inscripcion_inicial,
  CAST(antiguedad_anios AS DECIMAL(5,1))             AS antiguedad_anios,
  automotor_tipo_id, automotor_tipo_descripcion, categoria_vehiculo,
  automotor_marca,
  marca_txt                                          AS automotor_marca_original,
  automotor_marca_codigo_original, 
  automotor_modelo_codigo_original,
  automotor_modelo_descripcion, modelo_base,
  automotor_uso_id, 
  automotor_uso_descripcion,
  titular_tipo_persona, 
  titular_genero,
  CAST(titular_anio_nacimiento AS INT)               AS titular_anio_nacimiento,
  CAST(edad_titular AS INT)                          AS edad_titular,
  titular_pais_nacimiento, 
  titular_pais_nacimiento_id,
  titular_domicilio_localidad, 
  titular_domicilio_provincia, 
  titular_domicilio_provincia_id,
  CAST(titular_porcentaje_titularidad AS DOUBLE)     AS titular_porcentaje_titularidad,
  es_transferencia_parcial,
  hash_fila,
  -- Filas idénticas (flotas): se numeran 1,2,3... y se cuentan. NO se borran.
  CAST(ROW_NUMBER() OVER (PARTITION BY hash_fila ORDER BY _archivo_origen) AS INT) AS repeticion,
  CAST(COUNT(*)     OVER (PARTITION BY hash_fila) AS INT)                          AS filas_identicas,
  dq_problemas,
  (dq_problemas = '')                                AS fila_valida,
  _archivo_origen,
  CAST(_ingesta_ts AS TIMESTAMP)                     AS _ingesta_ts,
  CURRENT_TIMESTAMP()                                AS _processing_timestamp
FROM s;

### Revisar la vista antes de insertar (cada celda es un chequeo)

In [0]:
%sql
-- Chequeo 1: la vista no pierde ni inventa filas (esperado: igual que Bronze = 1.378.157)
SELECT (SELECT COUNT(*) FROM bootcamp.bronze.transferencias_autos_bronze) AS bronze,
       (SELECT COUNT(*) FROM transf_silver_prep)                           AS vista;

In [0]:
%sql
-- Chequeo 2: los tipos son los que queremos (DATE, INT, DECIMAL, DOUBLE, BOOLEAN...)
DESCRIBE QUERY SELECT * FROM transf_silver_prep;

In [0]:
%sql
-- Chequeo 3: qué problemas marca dq_problemas y cuántas filas afecta (hay filas con más de un problema)
SELECT problema, COUNT(*) AS filas
FROM (SELECT explode(split(dq_problemas, ',')) AS problema FROM transf_silver_prep WHERE dq_problemas <> '')
GROUP BY problema ORDER BY filas DESC;
-- Lo esperado según el EDA: anio_modelo_vacio ~7.078, edad_titular_fuera_de_rango ~469,
-- provincia_registro_vacia ~22.065, uso_vacio ~73, inscripcion_vacia 8, anio_nacimiento_vacio ~55.

In [0]:
%sql
-- Chequeo 4: reglas de personas jurídicas y edades (esperado: 0 y 0)
SELECT COUNT_IF(titular_tipo_persona = 'Jurídica' AND titular_anio_nacimiento IS NOT NULL) AS juridicas_con_anio_nac,
       COUNT_IF(edad_titular < 18 OR edad_titular > 110)                                  AS edades_imposibles
FROM transf_silver_prep;

In [0]:
%sql
-- Chequeo 5: rol_comerciante debe dar lo mismo que en el EDA (COMPRA 51.357 / VENDE 41.944 / SIN 1.284.856)
SELECT rol_comerciante, titular_tipo_persona, COUNT(*) AS filas
FROM transf_silver_prep GROUP BY ALL ORDER BY 1, 3 DESC;

In [0]:
%sql
-- Chequeo 6: marcas después de normalizar (VOLSKWAGEN, RENUALT, etc. ya no deberían aparecer)
SELECT automotor_marca, COUNT(*) AS filas
FROM transf_silver_prep GROUP BY 1 ORDER BY 2 DESC LIMIT 30;

In [0]:
%sql
-- Chequeo 7: qué quedó en 'OTRO' en categoria_vehiculo (para revisar a ojo)
SELECT automotor_tipo_descripcion, COUNT(*) AS filas
FROM transf_silver_prep WHERE categoria_vehiculo = 'OTRO'
GROUP BY 1 ORDER BY 2 DESC LIMIT 30;

In [0]:
%sql
-- Chequeo 8: flotas. Filas idénticas que se conservan numeradas (esperado: ~1.704 grupos con filas_identicas > 1)
SELECT COUNT(DISTINCT hash_fila) AS grupos, SUM(CASE WHEN repeticion > 1 THEN 1 ELSE 0 END) AS filas_extra
FROM transf_silver_prep WHERE filas_identicas > 1;
-- Esperado: ~1.704 grupos y ~2.536 filas extra (es lo mismo que midió el EDA con GROUP BY ALL)

In [0]:
%sql
-- Chequeo 9: la llave técnica (hash_fila + repeticion) NO puede repetirse. Esperado: 0 filas.
SELECT hash_fila, repeticion, COUNT(*) AS veces
FROM transf_silver_prep GROUP BY hash_fila, repeticion HAVING COUNT(*) > 1;

## Parte 3: Cargar Silver con MERGE (incremental e idempotente)

### Sobre tu pregunta: ¿un MERGE tipo SCD2?

**SCD tipo 2 no aplica a esta tabla, pero el MERGE sí.** Son dos cosas distintas:

- **SCD2** sirve para una *entidad que cambia en el tiempo* (un cliente que cambia de domicilio, una marca que cambia de nombre). Necesita una **clave natural** para reconocer "es el mismo, pero cambió" y guarda las versiones con `vigente_desde / vigente_hasta / es_actual`.
- Acá cada fila es un **hecho** (una transferencia registrada) que **no cambia nunca**, y además **no hay clave natural** (no hay patente ni id de trámite). No hay "versión anterior" que cerrar.

**Lo que sí hacemos:** `MERGE ... WHEN NOT MATCHED THEN INSERT`. Es lo que necesitamos de un SCD2 en la entrada: que **correr dos veces no duplique** y que **entren solo las filas nuevas**.
- La llave es `hash_fila + repeticion`. Así dos filas idénticas legítimas (la flota de 14 Hilux) conviven, y volver a correr el mismo mes inserta **0 filas**.
- No hay `WHEN MATCHED UPDATE` porque los hechos no se modifican. Si DNRPA corrigiera un archivo, se reemplaza el mes completo (ver celda opcional al final).

**Dónde sí iría SCD2:** en Gold, en una dimensión como `dim_registro_seccional` o `dim_marca_modelo` si querés conservar cómo cambian sus atributos (nombre, provincia, categoría) entre cargas.

In [0]:
%sql
MERGE INTO bootcamp.silver.transferencias_autos AS t
USING transf_silver_prep AS s
  ON  t.hash_fila  = s.hash_fila
  AND t.repeticion = s.repeticion
WHEN NOT MATCHED THEN INSERT *;
-- INSERT * mapea por NOMBRE de columna: por eso la vista y la tabla tienen exactamente los mismos nombres.
-- La primera vez inserta 1.378.157 filas.

## Parte 4: Validar la tabla Silver

### Ejercicio 4.1: Bronze vs Silver (esperado: iguales)

In [0]:
%sql
SELECT 'Bronze' AS capa, COUNT(*) AS registros FROM bootcamp.bronze.transferencias_autos_bronze
UNION ALL
SELECT 'Silver', COUNT(*) FROM bootcamp.silver.transferencias_autos;

### Ejercicio 4.2: Probar que es idempotente
Volvé a correr **la celda del MERGE** (Parte 3) y después esta celda: la fila del último `MERGE` tiene que mostrar `numTargetRowsInserted = 0` y el total de Silver no cambia.

In [0]:
%sql
SELECT version, timestamp, operation,
       operationMetrics['numTargetRowsInserted'] AS filas_insertadas,
       operationMetrics['numTargetRowsUpdated']  AS filas_actualizadas
FROM (DESCRIBE HISTORY bootcamp.silver.transferencias_autos)
WHERE operation = 'MERGE'
ORDER BY version;

### Ejercicio 4.3: Calidad de la tabla (equivalente al 3.4 de propiedades)

In [0]:
%sql
SELECT
  COUNT(*)                                                AS total_silver,
  COUNT_IF(fila_valida)                                   AS filas_validas,
  ROUND(COUNT_IF(fila_valida) * 100.0 / COUNT(*), 2)      AS pct_validas,
  COUNT_IF(tramite_fecha IS NULL)                         AS sin_fecha_tramite,
  COUNT_IF(automotor_anio_modelo IS NULL)                 AS sin_anio_modelo,
  COUNT_IF(antiguedad_anios IS NULL)                      AS sin_antiguedad,
  COUNT_IF(automotor_marca IS NULL)                       AS sin_marca,
  COUNT_IF(es_transferencia_parcial)                      AS transferencias_parciales,
  COUNT_IF(filas_identicas > 1)                           AS filas_en_grupos_identicos
FROM bootcamp.silver.transferencias_autos;

In [0]:
%sql
-- Los dq_problemas por tipo, ya sobre la tabla Silver
SELECT problema, COUNT(*) AS filas
FROM (SELECT explode(split(dq_problemas, ',')) AS problema FROM bootcamp.silver.transferencias_autos WHERE dq_problemas <> '')
GROUP BY problema ORDER BY filas DESC;

### Ejercicio 4.4: Primera consulta de negocio sobre Silver
Ranking mensual de marcas. Ahora con `COUNT(*)` (transferencias registradas), sin `peso`, y con la marca ya normalizada.

In [0]:
%sql
SELECT mes, automotor_marca, COUNT(*) AS transferencias,
       RANK() OVER (PARTITION BY mes ORDER BY COUNT(*) DESC) AS posicion
FROM bootcamp.silver.transferencias_autos
GROUP BY mes, automotor_marca
QUALIFY posicion <= 5
ORDER BY mes, posicion;

## Parte 5 (opcional): Cargar un mes nuevo o reprocesar uno

**Mes nuevo:** `COPY INTO` carga el CSV nuevo en Bronze → volvés a correr la vista (Parte 2) y el MERGE (Parte 3). Entran solo las filas nuevas.

**Reprocesar un mes** (por ejemplo si cambiás una regla o DNRPA corrige un archivo): borrás ese mes de Silver y volvés a correr el MERGE. Dejo la celda comentada para que no se corra por accidente.

In [0]:
%sql
-- DELETE FROM bootcamp.silver.transferencias_autos WHERE periodo_archivo = '202609';
-- Después: volver a correr la celda del MERGE (Parte 3).
SELECT 'celda opcional, está comentada' AS nota;

## Pendientes que quedan abiertos (del EDA)
1. Revisar lo que cae en `OTRO` (Chequeo 7) y unificar `CHASIS CON CABINA` / `BERLINA`.
2. Contar los `1900-01-01` y las 315 filas sin ID de país.
3. Interpretar `F17` y `C/PEDIDO` en `tramite_tipo`.
4. Confirmar con el diccionario del dataset qué es una fila y qué es Protocolo 21.
5. Test con filas inventadas con casos extremos (fechas raras, años 9999, porcentajes 0 o 150).